In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations

In [ ]:
# Load dataset
DIR = Path.cwd()

customers = pd.read_csv(DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DIR / "olist_orders_dataset.csv")
items_per_order = pd.read_csv(DIR / "olist_order_items_dataset.csv")
product_details = pd.read_csv(DIR / "olist_products_dataset.csv")

df = customers
df = df.merge(orders, on="customer_id")
df = df.merge(items_per_order, on="order_id")
df = df.merge(product_details, on="product_id")

# convert dates to datetime objects
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["order_approved_at"] = pd.to_datetime(df["order_approved_at"])
df["order_delivered_carrier_date"] = pd.to_datetime(df["order_delivered_carrier_date"])
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"])
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"])

# remove unecessary columns
df.drop([
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "order_status",
    "order_approved_at",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",],
    axis=1, inplace=True)

In [ ]:
df

In [ ]:
df.nunique()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
# What is the average order value?
df.groupby("order_id")["price"].sum().mean()

In [ ]:
# Which 10 products generate the most revenue?
df.groupby("product_id")["price"].sum().sort_values(ascending=False).head(10)

In [ ]:
# Top 5 months per orders
(
    df.groupby(df["order_purchase_timestamp"].dt.to_period("M"))["order_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(5)
)

In [ ]:
# Average delivery time in days
df["delivery_time_in_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / (24 * 3600)
average_delivery_time = float(df.drop_duplicates(subset=["order_id"])["delivery_time_in_days"].mean())
average_delivery_time

In [ ]:
# Average delivery time per month (based on the order purchase time)
df.groupby(df["order_purchase_timestamp"].dt.to_period("M"))["delivery_time_in_days"].mean().plot(kind="bar")

In [ ]:
# Total revenue
df["price"].sum()

In [ ]:
# Total items per order (top 50)
df["order_id"].value_counts().head(50).plot(kind="line")
plt.xticks([])

In [ ]:
# Average items per order
df["order_id"].value_counts().mean()

In [ ]:
# Percentage of orders per item quantity
df["order_id"].value_counts().value_counts().sort_index() / df["order_id"].nunique() * 100

In [ ]:
# How many order per item quantity
df["order_id"].value_counts().value_counts().sort_index().plot(kind="bar")
plt.yscale("log")

In [ ]:
# Scpecif items bought together
df.groupby(["order_id", "product_id"]).size().sort_values(ascending=False).head(20)

In [ ]:
# Count how many different items per order
df.groupby("order_id")["product_id"].nunique().sort_values(ascending=False).head(20)

In [ ]:
# Categories bought together
multi_orders = df.groupby("order_id")["product_category_name"].nunique()
multi_orders = multi_orders[multi_orders > 1].index

df_multi = df[df["order_id"].isin(multi_orders)]
df_multi.groupby(["order_id", "product_category_name"]).size().head(30)

pairs = []

for order_id, group in df_multi.groupby("order_id"):
    products = group["product_category_name"].dropna().unique()
    pairs += list(combinations(sorted(products), 2))

pair_counts = pd.Series(pairs).value_counts()
top_pairs = pair_counts.head(10)

labels = [(pair[0] + " + " + pair[1]) for pair in top_pairs.index]

top_pairs.plot(kind="bar")
plt.xticks(ticks=range(len(labels)), labels=labels)
plt.show()